In [ ]:
# deps
%pip install -q requests urllib3

import os, sys
import urllib3

# --- paths ---
LIB_PARENT = r"C:\Users\jaskew\Documents\project_repository\notebooks"  # parent of IsolatedTCPythonLibrary
if LIB_PARENT not in sys.path:
    sys.path.insert(0, LIB_PARENT)

# Ensure these exist (empty files are fine)
# C:\Users\jaskew\Documents\project_repository\notebooks\IsolatedTCPythonLibrary\__init__.py
# C:\Users\jaskew\Documents\project_repository\notebooks\IsolatedTCPythonLibrary\utils\__init__.py

# --- imports from your package ---
try:
    from IsolatedTCPythonLibrary import ThreatConnect, Owners, RequestObject
except ImportError:
    from IsolatedTCPythonLibrary.ThreatConnect import ThreatConnect
    from IsolatedTCPythonLibrary.Owners import Owners
    from IsolatedTCPythonLibrary.RequestObject import RequestObject

# config loader now lives inside the package
from IsolatedTCPythonLibrary.utils.config_loader import load_config

# --- config ---
CONFIG_PATH = os.path.join(
    LIB_PARENT, "IsolatedTCPythonLibrary", "utils", "config.json"
)
api_secret_key, api_access_id, api_base_url, api_default_org = load_config(CONFIG_PATH)
display(f"Loaded config: {CONFIG_PATH}")

# --- network options ---
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
VERIFY_SSL = False  # set True when using valid certs

# --- init ThreatConnect ---
tc = ThreatConnect(
    api_aid=api_access_id,
    api_sec=api_secret_key,
    api_org=api_default_org,
    api_url=api_base_url,
)
tc._verify_ssl = VERIFY_SSL
display("ThreatConnect initialized.")

# --- RequestObject ---
OWNER = "HTOC Org"
ro = RequestObject()
ro.set_http_method("GET")
ro.set_owner(OWNER)
ro.set_owner_allowed(True)
display("RequestObject ready.")

# --- sanity check: owners ---
owners = Owners(tc)
owners.retrieve_mine()
names = [getattr(o, "name", "") for o in owners._objects]
display({"owners_found": len(names), "names": names[:10]})


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ImportError: cannot import name 'ThreatConnect' from 'ThreatConnect' (consider renaming 'c:\\Users\\jaskew\\Documents\\project_repository\\notebooks\\IsolatedTCPythonLibrary.ipynb\\ThreatConnect.py' if it has the same name as a library you intended to import)

In [1]:
%pip install pandas

import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


'Querying owner: HTOC Org'

"Failed to query indicators for HTOC Org: name 'ro' is not defined"

'No valid indicator data found.'

""
